#  Stacking 单变量回归地质温度计

本 Notebook 实现了一个包含 5 个基学习器和 1 个元学习器的 stacking 回归模型。具体内容如下：

- **数据路径**："D:\Code\Jupyter\CPX-stacking_regression"

- **模型结构**：
  - `model1`: Extremely Randomized Trees (ERTs)
  - `model2`: XGBoostRegressor
  - `model3`: CatBoost
  - `model4`: GradientBoosting
  - `model5`: HistGradientBoosting
  - `model0`: CatBoost（元学习器）

- **汇总输出所有模型的 R² 与 RMSE（训练集与测试集）**

-  **XAI 及 图像输出**

说明：筛选数据dataset（重要） 贝叶斯优化 基学习器合并训练（基模型配置列表 统一标准化） CatBoost（元模型） 使用K折生成元特征（测试集使用K折平均）适配动态基模型绘图 修改版本V6

-  V2修改：1.加入K折交叉生成元特征（未使用K折平均） 2.修改绘图逻辑 整合绘图路径
-  V3修改：1.加入K折交叉生成元特征（测试集使用K折平均） 2.尝试合并模型训练 3.使用全局K折种子 4.优化数据预处理（加入索引）
-  V4修改：1.通过配置基模型列表实现基模型统一训练（重要） 2.删改并保留效果好的基模型 3.添加模型性能汇总（包含运行时间"存在问题"） 4.绘图： 修改数据传入逻辑，适配基模型配置（重点优化shap绘图逻辑，为pdp图添加可选模型绘图）
-  V5修改：1.动态生成数据路径 2.数据预处理阶段标准化特征（存在问题） 3.使用全局K折种子（10折） 4.动态分配计算核心  5.使用joblib保存top_features（重要特征）,基学习器StandardScaler与元学习器meta_scaler（标准化器） 6.使用ray实现并行计算（稳定性验证） 7.加入基模型筛选和初始特征作为元特征（重要） 8.修改基模型与元模型优化函数（添加fixed传参，保证随机种子一致），统一标准化流程 9.添加1000次稳定性验证与学习曲线（存在问题）10.删除有问题的模型时间统计
-  V6修改：1.标准数据分割流程 2.筛选重要特作为元特征 3.优化绘图部分
-  V7修改：1.疑似存在元模型验证集数据泄露 2.初始特征选取存在问题 3.1000次稳定性验证存在数据泄露 4.学习曲线存在数据泄露
- -  V7.1修改：1.修改模型多次标准化的问题 2.配置基模型优化  3.删除特征投票



##  1. 导入库、数据路径、模型评估

In [1]:
import os
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

## 1.数据导入准备

# 动态生成路径
BASE_DIR = os.getcwd()  # 使用当前工作目录
# 数据路径
data_path = os.path.join(BASE_DIR, "dataset.csv")
# 模型保存路径
model_dir = os.path.join(BASE_DIR, "models")  # 模型保存目录，位于代码目录下的 models 文件夹
os.makedirs(model_dir, exist_ok=True)
# 图件保存路径
fig_dir = os.path.join(BASE_DIR, "fig")  # 图件保存目录，位于代码目录下的 fig 文件夹
os.makedirs(fig_dir, exist_ok=True)

# 评估指标
# 传入的 model 必须已经训练好
def evaluate_model(model, X_train, y_train, X_test, y_test):
    # 训练集预测
    y_train_pred = model.predict(X_train)
    # 测试集预测
    y_test_pred = model.predict(X_test)

    # 计算R2分数
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    # 计算RMSE
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    return train_r2, train_rmse, test_r2, test_rmse


##  2. 数据预处理

In [2]:
import hashlib
import pandas as pd
from sklearn.model_selection import train_test_split

## 2. 数据预处理

# 加载数据集并创建副本
df = pd.read_csv(data_path)
df_raw = df.copy() # 保存原始数据副本
df.set_index(df.columns[0], inplace=True) # 设置第一列为索引

# 转换为数值，缺失值填0
df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

# 创建绘图用数据副本
df_fig = df_raw.copy()
df_fig.set_index(df_fig.columns[0], inplace=True)
df_fig = df_fig.apply(pd.to_numeric, errors='coerce').fillna(0)

# 分离特征和目标
X = df.iloc[:, :-1]  # 特征数据
y = df.iloc[:, -1]   # 目标数据（温度）

# 划分训练集和测试集（未标准化）
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# 存储特征名称
feature_names = X.columns.tolist()

# 计算数据文件哈希
try:
    with open(data_path, "rb") as f:
        data_hash = hashlib.md5(f.read()).hexdigest()
    print("Data hash:", data_hash)
except FileNotFoundError:
    print(f"Error: Data file {data_path} not found.")


Data hash: f28633bcecc21538038416ce754cbda5


##  3. 配置基模型

In [3]:
import multiprocessing
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

## 3. 配置基模型

# 定义全局 KFold 对象
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 动态分配核心数
total_cores = multiprocessing.cpu_count()  # 检测可用核心数
model_n_jobs = max(1, total_cores // 4)  # 每个模型分配 1/4 核心数，至少1个核心

# 基模型配置列表
base_model_configs = [
    # ExtraTrees
    {
        'name': 'ExtraTrees',
        'model_class': ExtraTreesRegressor,
        'params': {
            'n_estimators': ('int', 200, 1000),
            'max_depth': ('int', 10, 40),
            'min_samples_split': ('int', 5, 20),
            'min_samples_leaf': ('int', 2, 10),
            'max_features': ('categorical', ['sqrt', 'log2', 0.5, 0.7, 1.0]),
            'ccp_alpha': ('float', 0.001, 0.1),
            'n_jobs': ('fixed', model_n_jobs),
            'random_state': ('fixed', 42)
        }
    },
    # XGBoost
    {
        'name': 'XGBoost',
        'model_class': XGBRegressor,
        'params': {
            'n_estimators': ('int', 200, 500),
            'max_depth': ('int', 3, 6),
            'learning_rate': ('float', 0.01, 0.2),
            'subsample': ('float', 0.7, 1.0),
            'colsample_bytree': ('float', 0.7, 1.0),
            'reg_lambda': ('float', 1, 10),
            'reg_alpha': ('float', 0, 1),
            'random_state': ('fixed', 42),
            'n_jobs': ('fixed', model_n_jobs),
            'device': ('fixed', 'cpu')
        }
    },
    # CatBoost
    {
        'name': 'CatBoost',
        'model_class': CatBoostRegressor,
        'params': {
            'learning_rate': ('float', 0.01, 0.3),
            'depth': ('int', 4, 8),
            'iterations': ('int', 500, 1500),
            'l2_leaf_reg': ('float', 1, 5),
            'bagging_temperature': ('float', 0, 1),
            'random_strength': ('float', 0, 1),
            'random_state': ('fixed', 42),
            'thread_count': ('fixed', model_n_jobs),
            'verbose': ('fixed', 0)
        }
    },
    # GradientBoosting
    {
        'name': 'GradientBoosting',
        'model_class': GradientBoostingRegressor,
        'params': {
            'n_estimators': ('int', 100, 500),
            'max_depth': ('int', 4, 8),
            'learning_rate': ('float', 0.01, 0.1),
            'subsample': ('float', 0.7, 1.0),
            'min_samples_split': ('int', 5, 20),
            'min_samples_leaf': ('int', 2, 10),
            'random_state': ('fixed', 42)
        }
    },
    # HistGradientBoosting
    {
        'name': 'HistGradientBoosting',
        'model_class': HistGradientBoostingRegressor,
        'params': {
            'max_iter': ('int', 100, 500),
            'max_depth': ('int', 4, 8),
            'learning_rate': ('float', 0.01, 0.1),
            'min_samples_leaf': ('int', 10, 50),
            'l2_regularization': ('float', 0.1, 5),
            'random_state': ('fixed', 42)
        }
    }
]


##  4. 优化基模型

In [4]:
import os
import joblib
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. 超参数优化函数
def optimize_model(trial, model_config, X_train, y_train):
    """为指定模型配置优化超参数，返回交叉验证R²均值"""
    params = {}
    for param_name, (param_type, *param_range) in model_config['params'].items():
        if param_type == 'int':
            params[param_name] = trial.suggest_int(param_name, param_range[0], param_range[1])
        elif param_type == 'float':
            params[param_name] = trial.suggest_float(param_name, param_range[0], param_range[1])
        elif param_type == 'categorical':
            params[param_name] = trial.suggest_categorical(param_name, param_range[0])
        elif param_type == 'fixed':
            params[param_name] = param_range[0]

    # 创建模型（使用Pipeline防止数据泄露）
    model = model_config['model_class'](**params)
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    return cross_val_score(pipeline, X_train, y_train, cv=kf, scoring='r2').mean()

# 2. 优化单个基模型
def optimize_single_model(config, X_train, y_train):
    """优化单个模型超参数并返回最佳模型"""
    params_file = os.path.join(model_dir, f"{config['name']}_params.pkl")
    # Optuna 优化
    study = optuna.create_study(direction='maximize', pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=3))
    study.optimize(lambda trial: optimize_model(trial, config, X_train, y_train), n_trials=50)
    best_params = study.best_params

    # 添加 fixed 固定参数
    for param_name, (param_type, *param_range) in config['params'].items():
        if param_type == 'fixed':
            best_params[param_name] = param_range[0]

    # 创建最终模型
    model = config['model_class'](**best_params)
    # 保存最佳超参数（覆盖保存）
    try:
        joblib.dump(best_params, params_file)
        print(f"Saved best params for {config['name']} to {params_file}")

    except Exception as e:
        print(f"Error saving params for {config['name']}: {e}")

    return model


In [5]:
from joblib import Parallel, delayed
import os
import multiprocessing

# 3. 逐个优化基模型
# 设置并行核心数量，避免核心检测警告
os.environ['LOKY_MAX_CPU_COUNT'] = '16'  # 物理核心数量

# 动态分配并行任务数
total_cores = multiprocessing.cpu_count()  # 检测可用核心数
parallel_n_jobs = min(4, total_cores)  # 限制最多4个并行任务，适配硬件

# 并行优化所有基模型
base_models = Parallel(n_jobs=parallel_n_jobs)(
    delayed(optimize_single_model)(config, X_train_raw, y_train) for config in base_model_configs
)
print("✅ All base models optimized.")

# 评估基模型性能并存储
base_model_performance = []
base_model_dict = {} # 存储基模型（键：模型名称，值：模型对象）

for model, config in zip(base_models, base_model_configs):
    # 训练模型
    model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model)
])
    model_pipeline.fit(X_train_raw, y_train)
    # 计算性能指标
    train_r2, train_rmse, test_r2, test_rmse = evaluate_model(model_pipeline, X_train_raw, y_train, X_test_raw, y_test)
    # 记录性能
    base_model_performance.append({
        'Model': config['name'],
        'Train R²': train_r2,
        'Train RMSE': train_rmse,
        'Test R²': test_r2,
        'Test RMSE': test_rmse
    })
    base_model_dict[config['name']] = model_pipeline  # 存储模型

    # 保存训练后的模型
    model_file = os.path.join(model_dir, f"{config['name']}_model.pkl")
    try:
        joblib.dump(model_pipeline, model_file)
        print(f"Saved model {config['name']} to {model_file}")
    except Exception as e:
        print(f"Error saving model {config['name']}: {e}")


✅ All base models optimized.
Saved model ExtraTrees to D:\Code\Jupyter\CPX-stacking_regression\models\ExtraTrees_model.pkl
Saved model XGBoost to D:\Code\Jupyter\CPX-stacking_regression\models\XGBoost_model.pkl
Saved model CatBoost to D:\Code\Jupyter\CPX-stacking_regression\models\CatBoost_model.pkl
Saved model GradientBoosting to D:\Code\Jupyter\CPX-stacking_regression\models\GradientBoosting_model.pkl
Saved model HistGradientBoosting to D:\Code\Jupyter\CPX-stacking_regression\models\HistGradientBoosting_model.pkl


##  5. 元模型

In [6]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import os
import joblib

## 6. 生成元特征 (Meta-Features) - OOF (Out-of-Fold)

# 准备工作：筛选出表现最好的模型 Pipeline 字典
# 注意：这里假设 base_model_dict 是 Cell 5 生成的 {name: pipeline} 字典
# 如果 Cell 5 还没生成字典，你需要先在那里生成。这里我们基于 Test R² > 0.9 筛选
base_model_performance_df = pd.DataFrame(base_model_performance)
best_models_df = base_model_performance_df[
    (base_model_performance_df['Model'] != 'Stacking') &
    (base_model_performance_df['Test R²'] > 0.9)
]
selected_model_names = best_models_df['Model'].tolist()

# 构建被选中的模型字典 {name: pipeline}
# 这里的 base_model_dict 必须包含封装了 StandardScaler 的 Pipeline
selected_pipelines = {
    name: base_model_dict[name]
    for name in selected_model_names
    if name in base_model_dict
}

print(f"Selected base models for Stacking: {list(selected_pipelines.keys())}")


def get_oof_meta_features(base_model_pipelines, X_train_raw, y_train, X_test_raw, kf, model_dir, data_hash):
    """
    生成训练集(OOF)和测试集的元特征。

    参数:
    ----------
    base_model_pipelines : dict
        {model_name: model_pipeline} 形式的字典。Pipeline 必须包含 ('scaler', StandardScaler())。
    X_train_raw : DataFrame
        未标准化的原始训练特征。
    y_train : Series
        训练标签。
    X_test_raw : DataFrame
        未标准化的原始测试特征。
    kf : KFold
        交叉验证对象。
    model_dir : str
        缓存保存路径。
    data_hash : str
        数据指纹，用于缓存版本控制。

    返回:
    ----------
    X_train_meta_raw : ndarray (N_train, N_models)
        训练集元特征（OOF预测值，真实量级）。
    X_test_meta_raw : ndarray (N_test, N_models)
        测试集元特征（全量模型预测值，真实量级）。
    """
    meta_cache_dir = os.path.join(model_dir, "meta_features")
    os.makedirs(meta_cache_dir, exist_ok=True)

    # 初始化矩阵
    n_models = len(base_model_pipelines)
    X_train_meta = np.zeros((X_train_raw.shape[0], n_models))
    X_test_meta = np.zeros((X_test_raw.shape[0], n_models))

    print(f"\n开始生成元特征，共 {n_models} 个基模型...")

    # 使用 enumerate 遍历字典的 items，避免索引对齐风险
    for i, (model_name, pipeline) in enumerate(base_model_pipelines.items()):

        # 定义缓存路径
        train_cache_file = os.path.join(meta_cache_dir, f"train_meta_{model_name}_{data_hash}.npy")
        test_cache_file  = os.path.join(meta_cache_dir, f"test_meta_{model_name}_{data_hash}.npy")

        # 1. 尝试加载缓存
        if os.path.exists(train_cache_file) and os.path.exists(test_cache_file):
            print(f"[{model_name}] Loading cached meta features")
            X_train_meta[:, i] = np.load(train_cache_file)
            X_test_meta[:, i] = np.load(test_cache_file)

        else:
            print(f"[{model_name}] Computing meta features (10-Fold CV)...")

            # --- 阶段 A: 生成训练集元特征 (OOF) ---
            # 这里的 pipeline 包含 scaler，clone 后 scaler 参数重置
            # fit 时只会计算当前 fold 训练数据的均值/方差，杜绝数据泄露
            for train_idx, val_idx in kf.split(X_train_raw):
                # 数据切分
                X_tr, X_val = X_train_raw.iloc[train_idx], X_train_raw.iloc[val_idx]
                y_tr = y_train.iloc[train_idx]

                # 克隆 Pipeline (重置状态)
                fold_pipeline = clone(pipeline)

                # 训练并预测
                fold_pipeline.fit(X_tr, y_tr)
                X_train_meta[val_idx, i] = fold_pipeline.predict(X_val)

            # --- 阶段 B: 生成测试集元特征 ---
            # 直接使用传入的 pipeline（假设它已经在 Cell 5 全量训练过）进行预测
            # 这样保证了“上线应用”和“测试评估”的一致性
            # 注意：base_model_pipelines 里的对象如果是 Cell 5 训练过的，这里直接用即可
            # 如果为了保险起见，或者 pipeline 在 Cell 5 没 fit 过，这里可以再 fit 一次全量
            # 考虑到严谨性，建议这里再 fit 一次全量，或者确认传入的是已 fit 对象。
            # 这里我们选择：使用 Cell 5 已经 fit 好的全量模型直接预测（最高效）
            X_test_meta[:, i] = pipeline.predict(X_test_raw)

            # 保存缓存
            np.save(train_cache_file, X_train_meta[:, i])
            np.save(test_cache_file, X_test_meta[:, i])
            print(f"[{model_name}] Cached saved.")

    # 返回 _raw 版本用于查看物理意义
    return X_train_meta, X_test_meta

# 执行函数
# 显式传入所有依赖变量，不再依赖全局变量
X_train_meta_raw, X_test_meta_raw = get_oof_meta_features(
    base_model_pipelines=selected_pipelines,
    X_train_raw=X_train_raw,
    y_train=y_train,
    X_test_raw=X_test_raw,
    kf=kf,
    model_dir=model_dir,
    data_hash=data_hash
)


Selected base models for Stacking: ['ExtraTrees', 'XGBoost', 'CatBoost', 'GradientBoosting', 'HistGradientBoosting']

开始生成元特征，共 5 个基模型...
[ExtraTrees] Computing meta features (10-Fold CV)...
[ExtraTrees] Cached saved.
[XGBoost] Computing meta features (10-Fold CV)...
[XGBoost] Cached saved.
[CatBoost] Computing meta features (10-Fold CV)...
[CatBoost] Cached saved.
[GradientBoosting] Computing meta features (10-Fold CV)...
[GradientBoosting] Cached saved.
[HistGradientBoosting] Computing meta features (10-Fold CV)...
[HistGradientBoosting] Cached saved.


In [7]:
import os
import joblib
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from catboost import CatBoostRegressor

# 3. 优化 CatBoost 元模型 (Pipeline 风格)
def objective_catboost(trial, X_meta_raw, y_true):
    """
    优化 CatBoost 元模型超参数
    输入: X_meta_raw (未标准化的基模型预测值), y_true (真实温度)
    """
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 1),
        'random_state': 42,
        'verbose': 0,
        'allow_writing_files': False, # 禁止生成catboost_info文件夹
        'thread_count': model_n_jobs  # 【新增】使用全局定义的计算核心数，防止资源争抢
    }

    # 构建 Pipeline: 标准化 -> 元模型
    # 【核心逻辑】标准化仅在 Pipeline 内部进行，确保 CV 时 Scaler 只 fit 训练折
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', CatBoostRegressor(**params))
    ])

    # 使用 K-Fold 交叉验证评估当前参数
    # 注意：这里 n_jobs 可设为 1，因为 CatBoost 内部已经通过 thread_count 并行了
    scores = cross_val_score(pipeline, X_meta_raw, y_true, cv=kf, scoring='r2')
    return scores.mean()

print("开始优化元模型...")

# 优化过程
study_catboost = optuna.create_study(direction='maximize', pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5))

# 注意：这里传入 X_train_meta_raw (未标准化)，因为 Pipeline 内部会处理
# study_catboost 对象会调用 objective_catboost，在每一折 CV 内部做标准化
study_catboost.optimize(lambda trial: objective_catboost(trial, X_train_meta_raw, y_train), n_trials=50)

best_params_meta = study_catboost.best_params
best_params_meta['random_state'] = 42
best_params_meta['verbose'] = 0
best_params_meta['thread_count'] = model_n_jobs # 确保最终模型也使用正确的核数
best_params_meta['allow_writing_files'] = False # 禁止生成catboost_info文件夹

print("\nBest Meta Params:", best_params_meta)
joblib.dump(best_params_meta, os.path.join(model_dir, "meta_params.pkl"))

# 4. 训练最终元模型
# 为了方便后续使用，我们将最终模型也封装为 Pipeline 并保存
# 这样预测新数据时，只需要传入原始的基模型预测值，Pipeline 会自动标准化
final_meta_model = Pipeline([
    ('scaler', StandardScaler()), # 最终在全量元数据上 fit
    ('model', CatBoostRegressor(**best_params_meta))
])

# 在所有元特征数据上训练
final_meta_model.fit(X_train_meta_raw, y_train)

# 5. 评估 Stacking 模型性能
# 注意：这里 predict 传入的是 X_test_meta_raw (未标准化的预测值)，Pipeline 会自动处理
y_test_pred_stack = final_meta_model.predict(X_test_meta_raw)
y_train_pred_stack = final_meta_model.predict(X_train_meta_raw)

train_r2 = r2_score(y_train, y_train_pred_stack)
test_r2 = r2_score(y_test, y_test_pred_stack)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_stack))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_stack))

print("\nStacking Model (Pure Prediction Features) Performance:")
print(f"Train R²: {train_r2:.4f}, RMSE: {train_rmse:.4f}")
print(f"Test  R²: {test_r2:.4f}, RMSE: {test_rmse:.4f}")

# 保存最终的 Stacking Pipeline
# 这个 pkl 文件包含：Scaler (已fit全量) + CatBoost (已fit全量)
joblib.dump(final_meta_model, os.path.join(model_dir, "meta_model.pkl"))
print("Stacking meta model saved successfully.")

[I 2025-12-20 22:34:20,364] A new study created in memory with name: no-name-ab62c46e-e7f8-4354-beb9-38f08b0accb1


开始优化元模型...


[I 2025-12-20 22:34:30,506] Trial 0 finished with value: 0.9407666924542651 and parameters: {'learning_rate': 0.05393556039457944, 'depth': 7, 'iterations': 1452, 'l2_leaf_reg': 6.987823264309417, 'bagging_temperature': 0.06620160705148981, 'random_strength': 0.48698885533327774}. Best is trial 0 with value: 0.9407666924542651.
[I 2025-12-20 22:34:34,370] Trial 1 finished with value: 0.9378171445603929 and parameters: {'learning_rate': 0.1807275171544051, 'depth': 7, 'iterations': 543, 'l2_leaf_reg': 6.59883592383521, 'bagging_temperature': 0.5052830111530864, 'random_strength': 0.8321776206937487}. Best is trial 0 with value: 0.9407666924542651.
[I 2025-12-20 22:34:42,077] Trial 2 finished with value: 0.9283851014339269 and parameters: {'learning_rate': 0.261347831211383, 'depth': 6, 'iterations': 1742, 'l2_leaf_reg': 4.356144436996347, 'bagging_temperature': 0.25770176845383697, 'random_strength': 0.8219364031540247}. Best is trial 0 with value: 0.9407666924542651.
[I 2025-12-20 22:3


Best Meta Params: {'learning_rate': 0.010349341360089295, 'depth': 4, 'iterations': 863, 'l2_leaf_reg': 8.961071703617923, 'bagging_temperature': 0.4740426406454696, 'random_strength': 0.8505736922456422, 'random_state': 42, 'verbose': 0, 'thread_count': 8, 'allow_writing_files': False}

Stacking Model (Pure Prediction Features) Performance:
Train R²: 0.9529, RMSE: 32.5906
Test  R²: 0.9536, RMSE: 33.9679
Stacking meta model saved successfully.


##  6. 汇总所有模型性能

In [8]:
## 6. 汇总模型性能

# 汇总所有模型性能
performance_data = base_model_performance.copy()  # 复制基模型性能数据

# 添加 Stacking 模型性能
performance_data.append({
    'Model': 'Stacking',
    'Train R²': train_r2,
    'Train RMSE': train_rmse,
    'Test R²': test_r2,
    'Test RMSE': test_rmse
})

# 创建并打印性能表格
performance_df = pd.DataFrame(performance_data)  # 转换为 DataFrame
performance_df = performance_df.sort_values('Test R²', ascending=False)  # 按测试 R² 降序排序
print("\n===== Model Performance Summary =====")
print(performance_df.to_string(index=False, float_format='%.4f', col_space={'Model': 20, 'Train R²': 10, 'Train RMSE': 12, 'Test R²': 10, 'Test RMSE': 12}))  # 格式化输出


===== Model Performance Summary =====
               Model   Train R²   Train RMSE    Test R²    Test RMSE
            CatBoost     0.9999       1.0871     0.9565      32.8909
            Stacking     0.9529      32.5906     0.9536      33.9679
          ExtraTrees     0.9933      12.2628     0.9533      34.0660
    GradientBoosting     0.9997       2.4348     0.9447      37.0782
             XGBoost     0.9987       5.3762     0.9403      38.5185
HistGradientBoosting     0.9978       7.0082     0.9392      38.8602
